# WattWise — Heuristic (Genetic Algorithm) for Battery Dispatch

Minimize the daily electricity bill by scheduling battery charging and discharging
in each 30-minute slot under CEB's Time-of-Use tariff.

**Method:** Genetic Algorithm (heuristic) — compared with the DP (exact method).

---

## Stage 1 — Data Loading

Load the cleaned 30-minute profiles into separate per-day arrays for solar, demand, and price.

### 1.1 Imports and data-loading functions

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Constants used throughout the notebook
DATA_PATH = Path("cleaning/profiles_30min.csv")
SLOTS_PER_DAY = 48          # 30-minute slots
SLOT_HOURS = 0.5            # each slot is half an hour


def load_profiles(path=DATA_PATH):
    """Load the cleaned 30-minute profile data."""
    df = pd.read_csv(path)
    df["production_kWh"] = df["production_kWh"].astype(float)
    df["consumption_kWh"] = df["consumption_kWh"].astype(float)
    df["price_LKR_per_kWh"] = df["price_LKR_per_kWh"].astype(float)
    return df


def list_days(df):
    """Return all available dates in the dataset."""
    return sorted(df["date"].unique())


def get_day(df, date):
    """Return one day's data as clean 48-element NumPy arrays."""
    d = (
        df[df["date"] == date]
        .sort_values("slot_start")
        .reset_index(drop=True)
    )

    if len(d) != SLOTS_PER_DAY:
        raise ValueError(
            f"{date}: expected {SLOTS_PER_DAY} slots, got {len(d)}"
        )

    return {
        "date": date,
        "solar": d["production_kWh"].to_numpy(),    # kWh generated per slot
        "demand": d["consumption_kWh"].to_numpy(),  # kWh consumed per slot
        "price": d["price_LKR_per_kWh"].to_numpy(), # Rs/kWh for that slot
        "band": d["band"].to_numpy(),
        "slot": d["slot_start"].to_numpy(),
    }

### 1.2 Sanity check — load one day and inspect it

In [3]:
# --- basic sanity check ---
df = load_profiles()
print("Days available:", list_days(df))

day = get_day(df, "2026-09-11")
print(f"\nLoaded {day['date']}")
print(f"  slots       : {len(day['solar'])}")
print(f"  total solar : {day['solar'].sum():.2f} kWh")
print(f"  total demand: {day['demand'].sum():.2f} kWh")
print(f"  price levels: {sorted(set(day['price']))} Rs/kWh")

Days available: ['2026-09-10', '2026-09-11', '2026-09-13', '2026-09-16', '2026-09-17']

Loaded 2026-09-11
  slots       : 48
  total solar : 14.07 kWh
  total demand: 9.58 kWh
  price levels: [np.float64(33.0), np.float64(47.0), np.float64(106.0)] Rs/kWh
